In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
import re
import time
import pickle
import scipy.sparse as sp
import json

df = pd.read_parquet("/kaggle/input/notebooks/anymansince2005/eda-notebook/arxiv_ml_filtered.parquet")
df = df.reset_index(drop=True)
print(f"Loaded {len(df):,} papers")
print(df[["paper_id", "title", "abstract", "categories"]].head(2))

Loaded 110 papers
    paper_id                                              title  \
0  0704.0047  Intelligent location of simultaneously active ...   
1  0704.0050  Intelligent location of simultaneously active ...   

                                            abstract      categories  
0    The intelligent acoustic emission locator is...  [cs.NE, cs.AI]  
1    Part I describes an intelligent acoustic emi...  [cs.NE, cs.AI]  


In [2]:
def clean_text(text:str) -> str:
    """
    Cleaning for TF-IDF:
    -Lowercase
    -Remove LaTeX math blocks and special characters
    -Collapse Whitespace
    """
    if not isinstance(text,str):
        return ""
    
    text = text.lower()
    text = re.sub(r"\$\$.*?\$\$"," ",text, flags=re.DOTALL)
    text = re.sub(r"\$.*?\$", " ",text)
    text = re.sub(r"[^a-z0-9\s]"," ",text)
    text = re.sub(r"\s+", " ",text).strip()
    return text

def build_corpus(df: pd.DataFrame) -> pd.Series:
    """
    Combine title+abstract into one text field.
    Title is repeated 2X to give it more weight
    """
    
    corpus = (
        df["title"].apply(clean_text)+""+
        df["title"].apply(clean_text)+""+
        df["abstract"].apply(clean_text)
    )
    return corpus

df["corpus"] = build_corpus(df)
print("\nSample corpus entry : ")
print(df["corpus"].iloc[0][:300])


Sample corpus entry : 
intelligent location of simultaneously active acoustic emission sources part iintelligent location of simultaneously active acoustic emission sources part ithe intelligent acoustic emission locator is described in part i while part ii discusses blind source separation time delay estimation and locat


In [3]:
print("\nBuilding TF-IDF matrix ...")
start = time.time()

def build_vectorizer(n_docs: int) -> TfidfVectorizer:
    if n_docs < 1_000:
        min_df, max_df, max_feat = 1,1.0,5_000
    elif n_docs <10_000:
        min_df, max_df, max_feat = 2,0.95,15_000
    else:
        min_df, max_df, max_feat = 3,0.90,30_000
    print(f"\nVectorizer configured for {n_docs} docs:")
    print(f"min_df = {min_df}, max_df = {max_df}, max_features = {max_feat:,}")
    
    return TfidfVectorizer(
        max_features = max_feat,
        min_df = min_df,
        max_df = max_df,
        ngram_range = (1,2),
        sublinear_tf = True,
        stop_words = 'english',
    )

vectorizer = build_vectorizer(len(df))
tfidf_matrix = vectorizer.fit_transform(df["corpus"])

elapsed = time.time()-start
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Build Time         : {elapsed:.1f}s")
print(f"Matrix Sparsity    : {1-tfidf_matrix.nnz/(tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.4f}")


Building TF-IDF matrix ...

Vectorizer configured for 110 docs:
min_df = 1, max_df = 1.0, max_features = 5,000
TF-IDF matrix shape: (110, 5000)
Build Time         : 0.1s
Matrix Sparsity    : 0.9837


In [4]:
def get_recom_by_index(
    idx :int,
    tfidf_mat,
    df : pd.DataFrame,
    top_n : int=10,) -> pd.DataFrame :

    "Using linear_kernel"
    
    query_vec = tfidf_mat[idx]
    scores = linear_kernel(query_vec,tfidf_mat).flatten()
    scores[idx] = -1
    top_indices = scores.argsort()[::-1][:top_n]
    
    results = df.iloc[top_indices][
      ["paper_id", "title", "categories", "date"]
    ].copy()
    results["similiarity_score"] = scores[top_indices]
    return results.reset_index(drop=True)

def get_recom_by_title(
    title_query : str,
    tfidf_mat,
    vectorizer,
    df : pd.DataFrame,
    top_n : int=10,) -> pd.DataFrame:

    "For title/abstract snippet"
    
    cleaned = clean_text(title_query)
    query_vec = vectorizer.transform([cleaned])
    scores = linear_kernel(query_vec, tfidf_mat).flatten()
    top_indices = scores.argsort()[::-1][:top_n]

    results = df.iloc[top_indices][
      ["paper_id", "title", "categories", "date"]
    ].copy()
    results["similiarity_score"] = scores[top_indices]
    return results.reset_index(drop=True)

def get_recom_by_abstract(
    abstract : str,
    tfidf_mat,
    vectorizer,
    df : pd.DataFrame,
    top_n : int =10,) -> pd.DataFrame:

    "For raw abstract"
    
    return get_recom_by_title(abstract, tfidf_mat, vectorizer, df, top_n)

In [5]:
print("\n"+"="*60)
print("TEST 1 - Using existing paper index")
print("="*60)

sample_idx = 42
sample_paper = df.iloc[sample_idx]
print(f"\nQuery paper: {sample_paper['title']}")
print(f"Categories   : {sample_paper['categories']}")

recs = get_recom_by_index(sample_idx, tfidf_matrix, df, top_n=10)
print("\nTop 10 recommendations:")
print(recs[["title","similiarity_score","categories"]].to_string(index = True))

print("\n"+"="*60)
print("Test 2 - Using title query")
print("="*60)

query = "transformer attention mechanism natural language processing"
print(f"\nQuery : '{query}'")
recs2 = get_recom_by_title(query, tfidf_matrix, vectorizer, df, top_n=10)
print("\nTop 10 recommendations:")
print(recs2[["title","similiarity_score","categories"]].to_string(index = True))

print("\n"+"="*60)
print("Test 3 - recommend using abstract snippet")
print("="*60)

abstract_snippet = """
We propose a graph neural network approach for node classification in heterogenous
networks. Our method learns embeddings by aggregating information from multi-hop
neighborhoods using attention weights.
"""
print(f"\nQuery Abstract: '{abstract_snippet.strip()}'")
recs3 = get_recom_by_abstract(abstract_snippet, tfidf_matrix, vectorizer, df, top_n=10)
print("\nTop 10 recommendations:")
print(recs3[["title","similiarity_score","categories"]].to_string(index = True))


TEST 1 - Using existing paper index

Query paper: The Parameter-Less Self-Organizing Map algorithm
Categories   : ['cs.NE' 'cs.AI' 'cs.CV']

Top 10 recommendations:
                                                                                                title  similiarity_score                categories
0                               Traitement Des Donnees Manquantes Au Moyen De L'Algorithme De Kohonen           0.075771          [stat.AP, cs.NE]
1                        Statistical Mechanics of Nonlinear On-line Learning for Ensemble\n  Teachers           0.052407  [cs.LG, cond-mat.dis-nn]
2  Improved Neural Modeling of Real-World Systems Using Genetic Algorithm\n  Based Variable Selection           0.050790                   [cs.NE]
3      Fuzzy Artmap and Neural Network Approach to Online Processing of Inputs\n  with Missing Values           0.047405                   [cs.AI]
4                                                               Learning from compressed observatio

In [6]:
def precision_at_k(recom_cats, query_cats, k=10):
    hits = sum(
        1 for cats in recom_cats[:k]
        if set(cats) & set(query_cats)
    )
    return hits/k

def baseline_evaluation(df, tfidf_mat,sample_size=200, k=10):
    indices = np.random.choice(len(df),size = sample_size,replace=False)
    precisions = []
    
    for idx in indices:
        query_cats = df.iloc[idx]["categories"]
        recs = get_recom_by_index(idx, tfidf_mat, df,top_n=k)
        p_at_k = precision_at_k(recs["categories"].tolist(), query_cats, k)
        precisions.append(p_at_k)
    
    return {
        "mean_precision@k" : round(np.mean(precisions),4),
        "median_precision@k" : round(np.median(precisions),4),
        "std" : round(np.std(precisions),4),
        "min" : round(np.min(precisions),4),
        "max" : round(np.max(precisions),4),
        "k" : k,
        "sample_size" : sample_size,
    }
    
print("\n"+"="*60)
print("OFFLINE EVALUATION")
print("="*60)

np.random.seed(42)
metrics = baseline_evaluation(df,tfidf_matrix,sample_size=110,k=10)
print("\nBaseline TF-IDF Metrics -")
for key,val in metrics.items():
    print(f"{key:<25}:{val}")


OFFLINE EVALUATION

Baseline TF-IDF Metrics -
mean_precision@k         :0.48
median_precision@k       :0.5
std                      :0.2467
min                      :0.0
max                      :1.0
k                        :10
sample_size              :110


In [7]:
print("\n"+"="*60)
print("Saving model artifacts")
print("="*60)

sp.save_npz("/kaggle/working/tfidf_matrix.npz", tfidf_matrix)
print("Saved : tfidf_matrix.npz")

with open("/kaggle/working/tfidf_vectorizer.pkl","wb") as f:
    pickle.dump(vectorizer,f)
print("Saved : tfidf_vectorizer.pkl")

df[["paper_id","title","abstract","categories","date","corpus"]].to_parquet("/kaggle/working/arxiv_ml_with_corpus.parquet", index=False)
print("Saved : arxiv_ml_with_corpus.parquet")

with open("/kaggle/working/baseline_metrics.json","w") as f:
    json.dump(metrics, f, indent=2)
print("Saved : baseline_metrics.json")


Saving model artifacts
Saved : tfidf_matrix.npz
Saved : tfidf_vectorizer.pkl
Saved : arxiv_ml_with_corpus.parquet
Saved : baseline_metrics.json
